In [2]:
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modelos
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

# Clustering
from sklearn.cluster import KMeans, DBSCAN

# Balanceo
from imblearn.over_sampling import SMOTE

# Métricas
from sklearn.metrics import classification_report, roc_auc_score

# Interpretabilidad
import shap
from lime.lime_tabular import LimeTabularExplainer

import matplotlib.pyplot as plt

/opt/miniconda3/envs/tfg_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv("bs140513_032310.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'bs140513_032310.csv'

### Separación entre train y test

In [3]:
X = df.drop('fraud', axis=1)
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

NameError: name 'df' is not defined

## Escalado de los datos para clustering

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Feature engineering

## Empleo de K-means para la creación de features

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(X_train_scaled)

X_train['cluster'] = kmeans.predict(X_train_scaled)
X_test['cluster'] = kmeans.predict(X_test_scaled)

# Distancia al centroide
train_dist = kmeans.transform(X_train_scaled)
test_dist = kmeans.transform(X_test_scaled)

X_train['cluster_distance'] = train_dist.min(axis=1)
X_test['cluster_distance'] = test_dist.min(axis=1)

## Empleo de DBSCAN para la creación de características

In [ ]:
db = DBSCAN(eps=0.5, min_samples=5)

train_labels = db.fit_predict(X_train_scaled)
test_labels = db.fit_predict(X_test_scaled)

X_train['dbscan_outlier'] = (train_labels == -1).astype(int)
X_test['dbscan_outlier'] = (test_labels == -1).astype(int)

## SMOTE

In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# Modelos base

## RANDOM FORREST

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_res, y_train_res)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:,1]

## XGBOOST

In [ ]:
xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_train_res, y_train_res)

xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:,1]

## LIGHTGBM

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42
)

lgb_model.fit(X_train_res, y_train_res)

lgb_pred = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)[:,1]

# Evaluación de modelos base

In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f"\n{name}")
    print(classification_report(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_proba))


In [ ]:
evaluate_model("Random Forest", y_test, rf_pred, rf_proba)
evaluate_model("XGBoost", y_test, xgb_pred, xgb_proba)
evaluate_model("LightGBM", y_test, lgb_pred, lgb_proba)

# Stacking de modelos (ensemble learning)

## Crear dataset meta

In [ ]:
rf_train = rf.predict_proba(X_train_res)[:,1]
xgb_train = xgb.predict_proba(X_train_res)[:,1]
lgb_train = lgb_model.predict_proba(X_train_res)[:,1]

rf_test = rf.predict_proba(X_test)[:,1]
xgb_test = xgb.predict_proba(X_test)[:,1]
lgb_test = lgb_model.predict_proba(X_test)[:,1]

X_meta_train = pd.DataFrame({
    'rf': rf_train,
    'xgb': xgb_train,
    'lgb': lgb_train
})

X_meta_test = pd.DataFrame({
    'rf': rf_test,
    'xgb': xgb_test,
    'lgb': lgb_test
})

## Meta-model

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
meta_model.fit(X_meta_train, y_train_res)

stack_pred = meta_model.predict(X_meta_test)
stack_proba = meta_model.predict_proba(X_meta_test)[:,1]

## Evaluación modelo de stacking

In [ ]:
evaluate_model("STACKING", y_test, stack_pred, stack_proba)

# Interpretabilidad

## SHAP de LightGBM

In [ ]:
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test)

### Interpretabilidad global

In [ ]:
shap.summary_plot(shap_values, X_test)

### Interpretabilidad local

In [ ]:
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test.iloc[0]
)

## LIME

In [ ]:
explainer_lime = LimeTabularExplainer(
    X_train.values,
    feature_names=X.columns,
    class_names=['No Fraud', 'Fraud'],
    mode='classification'
)

exp = explainer_lime.explain_instance(
    X_test.iloc[0].values,
    lgb_model.predict_proba
)

exp.show_in_notebook()

# Análisis clustering

In [ ]:
df['cluster'] = kmeans.predict(scaler.transform(X))
df['dbscan_outlier'] = (DBSCAN(eps=0.5).fit_predict(scaler.transform(X)) == -1).astype(int)

df.groupby('cluster')['fraud'].mean()
df.groupby('dbscan_outlier')['fraud'].mean()